Bu notebook tamamen prediction / demo için:
- Eğitilmiş modelleri .npz dosyalarından yüklüyoruz.
- Her dataset için class_names.txt’i okuyoruz.
- Bir yaprak fotoğrafı yolu verip 3 modelden de tahmin alıyoruz.


# Logistic Regression Models - Prediction Demo

Bu notebook, daha önce eğitilmiş Logistic Regression modellerini
(PlantVillage, Plant Disease Detection, PlantDoc Converted) yükler ve
verilen bir yaprak görüntüsü için her modelden tahmin (predicted class)
üretir.

Tüm eğitim, `02_logistic_regression_training.ipynb` içinde yapılmıştır.
Burada sadece:
- Modelleri `.npz` dosyalarından yüklüyoruz,
- `class_names.txt` dosyalarından sınıf isimlerini okuyoruz,
- Bir test görüntüsü üzerinde tahmin yapıyoruz.


# Cell 1 – Importlar

In [ ]:
import os
import math
import random
import numpy as np
from PIL import Image



# Cell 2 – Logistic Regression sınıfı (sadece prediction için yeterli)

In [ ]:
def softmax(logits):
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    s = sum(exps)
    if s == 0.0:
        c = len(logits)
        return [1.0 / c for _ in range(c)]
    return [e / s for e in exps]


class MulticlassLogisticRegression:
    def __init__(self, num_features, num_classes, learning_rate=0.1):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        self.W = [
            [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]
            for _ in range(num_features)
        ]
        self.b = [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]

    def _compute_logits(self, x):
        logits = [0.0 for _ in range(self.num_classes)]
        for k in range(self.num_classes):
            s = 0.0
            for j in range(self.num_features):
                s += x[j] * self.W[j][k]
            s += self.b[k]
            logits[k] = s
        return logits

    def predict_proba_one(self, x):
        logits = self._compute_logits(x)
        probs = softmax(logits)
        return probs

    def predict_one(self, x):
        probs = self.predict_proba_one(x)
        best_class = 0
        best_prob = probs[0]
        for k in range(1, self.num_classes):
            if probs[k] > best_prob:
                best_prob = probs[k]
                best_class = k
        return best_class

    @classmethod
    def load(cls, path):
        data = np.load(path)

        num_features = int(data["num_features"])
        num_classes = int(data["num_classes"])

        model = cls(
            num_features=num_features,
            num_classes=num_classes,
            learning_rate=0.01,
        )

        W_array = data["W"]
        b_array = data["b"]

        model.W = W_array.tolist()
        model.b = b_array.tolist()

        print(f"Model loaded from {path}")
        print("num_features:", num_features)
        print("num_classes:", num_classes)

        return model



# Cell 3 – Görseli 32×32 gri vektöre çeviren fonksiyon

In [ ]:
IMG_SIZE = 32

def load_image_as_vector(path, img_size=IMG_SIZE):
    """
    Load an image file, convert to grayscale 32x32, normalize and flatten.
    """
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())
        vector = [p / 255.0 for p in pixels]
        return vector



# Cell 4 – Sınıf isimlerini yükleme

In [ ]:
def load_class_names(base_dir):
    """
    Load class names from class_names.txt in the given preprocessed directory.
    """
    class_names = []
    class_names_path = os.path.join(base_dir, "class_names.txt")
    with open(class_names_path, "r", encoding="utf-8") as f:
        for line in f:
            class_names.append(line.strip())
    return class_names



# Cell 5 – Modelleri ve sınıf isimlerini yükle

In [ ]:
# Preprocessed directories (same as training notebook)
PV_DIR = "preprocessed_plantvillage"
PDD_DIR = "preprocessed_pdd"
PD_DIR = "preprocessed_plantdoc"

# Models directory
MODELS_DIR = "models"

# Load class names
class_names_pv = load_class_names(PV_DIR)
class_names_pdd = load_class_names(PDD_DIR)
class_names_pd = load_class_names(PD_DIR)

print("PlantVillage classes:", len(class_names_pv))
print("Plant Disease Detection classes:", len(class_names_pdd))
print("PlantDoc Converted classes:", len(class_names_pd))

# Load models
model_pv = MulticlassLogisticRegression.load(
    os.path.join(MODELS_DIR, "logreg_plantvillage.npz")
)
model_pdd = MulticlassLogisticRegression.load(
    os.path.join(MODELS_DIR, "logreg_plantdiseasedetection.npz")
)
model_pd = MulticlassLogisticRegression.load(
    os.path.join(MODELS_DIR, "logreg_plantdoc_converted.npz")
)



# Cell 6 – Prediction fonksiyonları

In [ ]:
def predict_with_model(image_path, model, class_names, model_label=""):
    """
    Predict class for a given image using the specified model and class names.
    """
    x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
    pred_idx = model.predict_one(x_vec)
    if 0 <= pred_idx < len(class_names):
        class_name = class_names[pred_idx]
    else:
        class_name = f"unknown_{pred_idx}"

    print("-" * 60)
    if model_label:
        print(f"Model: {model_label}")
    print(f"Image : {image_path}")
    print(f"Predicted class index: {pred_idx}")
    print(f"Predicted class name : {class_name}")
    return pred_idx, class_name



# Cell 7 – Test görseli ile demo

In [ ]:
# Example test image path (adjust this to your own test image)
# You can place a test image under e.g. "../TestImages/apple_leaf.jpg"
test_image_path = "../TestImages/sample_leaf.jpg"  # <-- change this

print("Prediction demo for a single leaf image:\n")

pv_idx, pv_name = predict_with_model(
    test_image_path,
    model_pv,
    class_names_pv,
    model_label="PlantVillage Logistic Regression",
)

pdd_idx, pdd_name = predict_with_model(
    test_image_path,
    model_pdd,
    class_names_pdd,
    model_label="Plant Disease Detection Logistic Regression",
)

pd_idx, pd_name = predict_with_model(
    test_image_path,
    model_pd,
    class_names_pd,
    model_label="PlantDoc Converted Logistic Regression",
)

